<a href="https://colab.research.google.com/github/bukunmibalogun5-eng/Training-LLMs/blob/main/Bigram.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as F
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)
block_size = 8
batch_size = 4
max_iters = 1000
# eval_interval = 2500
learning_rate = 3e-4
eval_iters = 250

cuda


In [3]:
with open('wizard_of_oz.txt', 'r', encoding='utf-8') as f:
    text = f.read()
chars = sorted(set(text))
print(chars)
vocab_size = len(chars)

['\n', ' ', '!', '"', '&', "'", '(', ')', '*', ',', '-', '.', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', '[', ']', '_', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '\ufeff']


In [4]:
string_to_int = { ch:i for i,ch in enumerate(chars) }
int_to_string = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [string_to_int[c] for c in s]
decode = lambda l: ''.join([int_to_string[i] for i in l])

data = torch.tensor(encode(text), dtype=torch.long)
# print(data[:100])

In [5]:
n = int(0.8*len(data))
train_data = data[:n]
val_data = data[n:]

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

x, y = get_batch('train')
print('inputs:')
# print(x.shape)
print(x)
print('targets:')
print(y)

inputs:
tensor([[55, 71, 58, 54, 73, 61,  1, 62],
        [73, 74, 71, 58, 11,  0,  0, 44],
        [ 0,  0,  3, 43, 62, 71,  9,  3],
        [75, 58, 71,  1, 66, 78,  1, 61]], device='cuda:0')
targets:
tensor([[71, 58, 54, 73, 61,  1, 62, 67],
        [74, 71, 58, 11,  0,  0, 44, 61],
        [ 0,  3, 43, 62, 71,  9,  3,  1],
        [58, 71,  1, 66, 78,  1, 61, 58]], device='cuda:0')


In [6]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [7]:
class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, index, targets=None):
        logits = self.token_embedding_table(index)


        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, index, max_new_tokens):
        # index is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # get the predictions
            logits, loss = self.forward(index)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            index_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            index = torch.cat((index, index_next), dim=1) # (B, T+1)
        return index

model = BigramLanguageModel(vocab_size)
m = model.to(device)

context = torch.zeros((1,1), dtype=torch.long, device=device)
generated_chars = decode(m.generate(context, max_new_tokens=500)[0].tolist())
print(generated_chars)


O4j ,o.BY,(L.BMqiox

bstV_xzzSJrN,UuA]tc'KK1﻿4MeDf1VrN.gTEBD(Wz4f9*Ytu4uC'yaZ8v]37Gw-G-GsEs-pQ49UDdK;4;M9(&A﻿pvB'nrN"lPfUh
&x&F;cOL!oC9*;eK
O&oQV1a3peug:HSNBrUo9y7Kyz&6vorC;﻿]jqwSy:zzP4u[uBXgWkv-s1BiB)PQ5MjkS)76cdY0PAW,jy:jjI[*dba!2udfVHvAEOXnBkN4MHHH(c-zXzOml3bpl
1Z3)CRRw(8
7*TJh[P7"6
1spJ;frmBT 3l'G(y8qIi"u[u,mEtP (uyJT8p;&2w,PJwuULMIFMl-T)?j
lF.Ir"JqD;JrkV28d-FqSy:"FG25_5
u:﻿]R
hP ,U-zstrq9rhSEc.bNrPYBFPfd]5KfmQAbRZH_0Ve0('*h pd6&](beDd2L!kqJQoQj'DwWsFV4qM4BG(E.rqAB('vY-IS!gYk2GB_q5l' pFKdxDh


In [8]:
# create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):
    if iter % eval_iters == 0:
        losses = estimate_loss()
        print(f"step: {iter}, train loss: {losses['train']:.3f}, val loss: {losses['val']:.3f}")

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model.forward(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
print(loss.item())

step: 0, train loss: 4.921, val loss: 4.913
step: 250, train loss: 4.873, val loss: 4.844
step: 500, train loss: 4.808, val loss: 4.792
step: 750, train loss: 4.718, val loss: 4.736
4.613348960876465


In [9]:
context = torch.zeros((1,1), dtype=torch.long, device=device)
generated_chars = decode(m.generate(context, max_new_tokens=500)[0].tolist())
print(generated_chars)


UreQf[grNB[7"(*l,,G﻿?juaG-w*XOF  VN;?zEwS]mBD:'RtbR5*73M8BXpum:PfbsknC;Uku1nW[sH(u-iT8q29x4uvMR3kDRMsk-﻿]PhMCV4!o.rsp:wT!!P-aK-QP﻿cglFC-Gwwh,:?7-VQffCqA4*m,w2WMd NC!oMJTF&Jwnjkt:Av ZHhMandK8G*u64drki
wf05QjqiK) E1Oa8KacJwXd2)6qz7O4!mpgny .Uq2-w-qKE.trH(Bp6
5YqFv6nYVe.qiVWumrOskOXh_6aTFB;LezO&Pfn',XTQ2skSJwa3zwk]]Rmnz&*)hoD,8oZT(F9'iWspbJva!QnCR﻿(KJF_0KM9*k_KsI7KaU4!B(QGw-1m:﻿b5_ChME-QJ0OCclbCV)Gvd)Iw-QObDdvhoofCTQjTjpBOXJeM2DdWfK8Ayzh;joJo*Ab00soTt
j!B-&Jw4CH(**Mu.BG1mF1mIx5Io"j_iu*s'WvO!0(U329B
